# 02. 서브그래프(Subgraphs)

> 큰 그래프는 작은 그래프를 합쳐 만들어요. **공유 키 / 독립 스키마** 두 가지 서브그래프 패턴과 3계층 중첩까지 다루며 모듈화 전략을 익혀요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **공유 키(Shared Keys) 패턴**으로 컴파일된 서브그래프를 부모 그래프의 노드로 직접 추가할 수 있어요
2. **독립 스키마(Independent Schemas) 패턴**으로 래퍼(wrapper) 함수를 작성해 두 그래프의 상태를 변환할 수 있어요
3. **3계층 서브그래프 구조**(grandchild → child → parent)를 설계하고 구현할 수 있어요
4. `subgraphs=True` 옵션으로 중첩된 서브그래프의 내부 실행 흐름을 스트리밍으로 확인할 수 있어요

## 사전 지식

- `StateGraph`, `TypedDict`, `add_node`, `add_edge` 기본 사용법 (Part 2 참고)
- 이전 노트북 `01-Functional-API.ipynb`에서 배운 `@entrypoint`, `@task` 함수형 API


## 서브그래프란?

서브그래프(Subgraph)는 **그래프 안에 또 다른 그래프를 포함**하는 구조예요. 복잡한 워크플로우를 독립적인 모듈로 나누거나, 멀티 에이전트 시스템에서 각 에이전트를 별도의 그래프로 구성할 때 매우 유용해요.

### 왜 서브그래프를 사용하나요?

레고 블록을 상상해보세요. 작은 블록들을 조합해서 큰 구조물을 만들 듯, 작은 그래프를 조합해서 복잡한 시스템을 만들어요. 각 서브그래프는 **독립적으로 개발/테스트/재사용**할 수 있어서 코드 관리가 훨씬 쉬워져요.

서브그래프를 추가하는 방법은 **두 그래프가 상태(State)를 어떻게 공유하느냐**에 따라 달라져요:

| 패턴 | 상태 관계 | 구현 방법 | 비유 |
|------|----------|----------|------|
| **Case 1: 공유 키** | 부모-자식이 동일한 키를 가짐 | 컴파일된 서브그래프를 `add_node`에 직접 전달 | 같은 언어를 쓰는 팀 (직접 대화) |
| **Case 2: 독립 스키마** | 부모-자식이 완전히 다른 키 사용 | 래퍼 함수를 노드로 등록하고 그 안에서 `invoke()` 호출 | 다른 언어를 쓰는 팀 (통역사 필요) |

> 🔑 **핵심 개념**: 서브그래프에서 **공유되지 않는 키**는 부모 그래프로 전파되지 않아요. 이를 **프라이빗 상태(Private State)**라고 해요. 자식 그래프 내부에서만 쓰이는 임시 키를 분리할 수 있어서 각 레이어의 독립성을 보장해요.


### 잠깐: 서브그래프와 서브에이전트는 무엇이 다를까요?

두 용어가 비슷하게 들리지만, 기준점이 달라요.

| 구분 | 서브그래프(Subgraph) | 서브에이전트(Subagent) |
|------|----------------------|-------------------------|
| 핵심 관점 | **그래프 구조**를 나누고 합치는 구현 단위 | **작업 역할**을 나누어 위임하는 에이전트 패턴 |
| 대표 형태 | `StateGraph`를 컴파일한 그래프를 부모 그래프의 노드로 사용 | 감독자(supervisor) 에이전트가 전문 에이전트를 도구처럼 호출 |
| 상태 공유 | 공유 키/프라이빗 키처럼 **State 채널** 기준으로 연결 | 보통 별도 메시지/컨텍스트 창에서 실행하고 결과만 감독자에게 반환 |
| 디버깅 관점 | 정적으로 연결된 경우 `subgraphs=True`로 내부 실행 흐름 추적 가능 | 도구 함수 안에서 호출하면 중첩 State가 자동으로 보이지 않을 수 있음 |
| 관계 | 서브에이전트를 구현하는 한 방법이 될 수 있음 | 내부 구현이 LangGraph 서브그래프일 수도 있고 아닐 수도 있음 |

> 한 문장으로 정리하면, **서브그래프는 그래프를 모듈화하는 방법**이고, **서브에이전트는 일을 전문 역할에게 위임하는 설계 패턴**이에요. 예를 들어 “배송 전문 에이전트”는 서브에이전트이고, 그 에이전트의 내부 절차가 `StateGraph`로 만들어져 부모 그래프에 연결되면 서브그래프이기도 합니다.


### 전체 아키텍처

```mermaid
flowchart TD
    subgraph PARENT["부모 그래프 (ParentState)"]
        P1["node_1"] --> C1
        subgraph SUBGRAPH_NODE["서브그래프 노드"]
            C1["subgraph_node_1"] --> C2["subgraph_node_2"]
        end
        C2 --> PE["END"]
    end

    A["입력"] --> P1
    PE --> B["출력"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A input
    class P1,C1,C2 process
    class PE,B output
    style SUBGRAPH_NODE fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```


## 0. 환경 설정


In [ ]:
# 환경 변수를 .env 파일에서 불러와요
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# LangSmith 추적 설정 - 그래프 실행 흐름을 온라인에서 확인할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Subgraphs"


## 1. Case 1: 공유 키(Shared Keys) 패턴

부모 그래프와 서브그래프가 **동일한 상태 키를 공유**하는 가장 일반적인 패턴이에요. 예를 들어 멀티 에이전트 시스템에서 모든 에이전트가 `messages` 키를 공유하는 경우가 대표적이에요.

### 구현 단계

1. 서브그래프 상태(`ChildState`)와 노드 함수를 정의한 뒤 컴파일해요
2. 부모 그래프에서 `builder.add_node("node_2", subgraph)`처럼 **컴파일된 서브그래프 객체를 직접 전달**해요

> 🔑 **핵심 개념**: `ChildState`에는 부모와 **공유하는 키**(name)와 **서브그래프 전용 키**(family_name)가 함께 있어요. LangGraph는 공유 키만 부모 ↔ 자식 사이에서 자동으로 전달하고, 전용 키는 서브그래프 내부에만 머물러요.

### 공유 키는 어떻게 결정되나요?

LangGraph에서 부모 그래프와 서브그래프를 `builder.add_node("node_2", subgraph)`처럼 **직접 연결**할 때, 공용 채널은 State 클래스 이름이 아니라 **상태 딕셔너리의 키 이름**으로 결정돼요.

- `ParentState.name`과 `ChildState.name`처럼 **키 이름이 같으면** 같은 state channel로 취급돼요.
- `ChildState.family_name`처럼 부모 State에 없는 키는 **서브그래프 내부 전용 키**예요.
- 두 State 전체가 같을 필요는 없어요. **공유할 키만 일치**하면 되고, 자식은 내부 처리용 키를 추가로 가질 수 있어요.
- 키 이름이 다르면 자동 공유가 아니므로, Case 2처럼 래퍼 함수에서 `부모 상태 → 자식 입력 → 부모 출력`을 직접 매핑해야 해요.
- 같은 이름의 키는 서로 값을 주고받는 통로이므로, 타입과 reducer 의미도 맞춰 두는 것이 안전해요.


In [ ]:
from langgraph.graph import START, END, StateGraph
from typing import TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: LangGraph의 핵심 그래프 구성 요소를 가져와요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 서브그래프 노드 함수 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 서브그래프 시각화

`xray=True` 옵션을 사용하면 내부 노드 구조까지 상세하게 볼 수 있어요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → subgraph_node_1 → subgraph_node_2 → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 부모 그래프 정의

이제 서브그래프를 포함하는 부모 그래프를 만들어요. 핵심은 `builder.add_node("node_2", subgraph)`처럼 **컴파일된 서브그래프 객체를 직접 전달**하는 거예요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 부모 그래프 상태 정의: 서브그래프(ChildState)와 name 키를 공유해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → node_1 → node_2(서브그래프) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 그래프 실행 — 기본 스트리밍

기본 스트리밍에서는 **상위 그래프 노드의 출력만** 보여줘요. 서브그래프 내부의 중간 단계는 표시되지 않아요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 스트리밍: 부모 그래프 노드 출력만 표시
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 그래프 실행 — subgraphs=True 옵션

`subgraphs=True`를 추가하면 서브그래프 내부의 중간 단계까지 모두 볼 수 있어요. 출력 형식이 `(네임스페이스 튜플, 상태 딕셔너리)` 형태로 변경돼요.

- 빈 튜플 `()` = 부모(상위) 그래프에서 발생한 이벤트
- `('node_2:...',)` = 서브그래프 내부에서 발생한 이벤트


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: subgraphs=True: 서브그래프 내부 단계까지 모두 표시
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. Case 2: 독립 스키마(Independent Schemas) 패턴

부모 그래프와 서브그래프의 상태 키가 **완전히 다른 경우**예요. 이때는 컴파일된 서브그래프를 직접 노드로 추가할 수 없어요.

대신 **래퍼(wrapper) 함수**를 노드로 등록하고, 그 안에서 두 가지 변환을 수행해요:

1. **호출 전**: 부모 상태 -> 서브그래프 입력 형식으로 변환
2. **호출 후**: 서브그래프 출력 -> 부모 상태 형식으로 변환

### Case 1 vs Case 2 비교

| 비교 항목 | Case 1: 공유 키 | Case 2: 독립 스키마 |
|-----------|----------------|---------------------|
| 구현 난이도 | 쉬움 (직접 전달) | 중간 (래퍼 함수 필요) |
| 상태 변환 | 자동 (LangGraph가 처리) | 수동 (개발자가 구현) |
| 재사용성 | 공유 키가 일치해야 함 | 어떤 서브그래프든 연결 가능 |
| 사용 시점 | 같은 도메인의 에이전트 조합 | 기존 독립 모듈 통합 |


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Case 2: 독립 스키마 서브그래프 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 부모 그래프 상태: 서브그래프와 공유하는 키가 전혀 없어요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → node_1 → node_2(래퍼) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 독립 스키마 그래프 실행 + 내부 흐름 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 3계층 서브그래프 구조 (grandchild → child → parent)

실제 복잡한 시스템에서는 서브그래프가 더 깊게 중첩될 수 있어요. 이번에는 **3단계 계층 구조**를 구현해볼게요.

각 계층은 완전히 독립적인 상태 키를 사용해요:

| 계층 | 상태 키 | 역할 |
|------|----------|------|
| `GrandChildState` | `my_grandchild_key` | 가장 안쪽 그래프 |
| `ChildState` | `my_child_key` | 중간 그래프, grandchild를 래퍼로 호출 |
| `ParentState` | `my_parent_key` | 최상위 그래프, child를 래퍼로 호출 |

```mermaid
flowchart LR
    subgraph PARENT["부모 그래프"]
        PA["parent_1"] --> CB["call_child_graph"] --> PB["parent_2"]
        subgraph CHILD["자식 그래프 (래퍼 내부)"]
            CB2["call_grandchild_graph"]
            subgraph GC["손자 그래프 (래퍼 내부)"]
                GCA["grandchild_1"]
            end
            CB2 --> GC
        end
    end

    IN["입력"] --> PA
    PB --> OUT["출력"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class IN input
    class PA,CB,PB,CB2,GCA process
    class OUT output
```

> 🔑 **핵심 개념**: 계층이 깊어져도 패턴은 동일해요. **각 계층은 자신이 직접 호출하는 바로 아래 계층만 알면 돼요**. grandchild 그래프는 parent가 존재하는지도 몰라도 돼요. 이것이 모듈화의 핵심이에요.


### 3-1. Grandchild 그래프


In [ ]:
from typing_extensions import TypedDict as TdDict  # 이미 TypedDict를 가져왔으므로 별칭 사용

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 손자 그래프의 상태와 노드를 정의해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-2. Child 그래프


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 자식 그래프 상태: my_child_key만 사용해요 (grandchild_key와 다른 키!)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-3. Parent 그래프 (최상위)


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 부모 그래프 상태: my_parent_key만 사용해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → parent_1 → child(래퍼) → parent_2 → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3계층 전체 실행 — subgraphs=True 로 흐름 추적

`subgraphs=True`로 실행하면 `parent → child → grandchild` 세 계층 모두의 실행 순서를 볼 수 있어요. 네임스페이스 튜플의 **길이**가 계층 깊이를 나타내요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3계층 그래프 전체 실행 - 각 계층의 흐름을 추적해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3계층 그래프 최종 결과 확인


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: invoke()로 최종 상태만 가져올 수도 있어요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. TODO 실습

아래 코드는 번역 서브그래프와 감정 분석 서브그래프를 조합하는 파이프라인이에요. 두 서브그래프는 독립적인 스키마를 가지고 있어요. 래퍼 함수를 완성해보세요!


In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

model = init_chat_model("openai:gpt-4o-mini")

# ============================================================
# TODO: translate_node 래퍼 함수를 완성하세요
# 힌트:
#   1. state["english_text"]를 translator_graph에 {"text": ...} 형식으로 전달하세요
#   2. 반환값의 "translated"를 꺼내서 {"korean_text": ...}로 부모 상태를 업데이트하세요
# 예상 결과: "Hello World" → "안녕 세계"
# ============================================================

    # TODO: 아래 두 줄을 완성하세요
    # translator_input = ???
    # result = ???
    # return ???


---

## 5. 가상의 실무 시나리오 — 고객 지원 티켓 라우터

마지막으로 **학생이 위에서 아래로 그대로 실행하며 따라 할 수 있는** 실무형 예제를 만들어볼게요. 이번 예제는 고객 문의를 받아서 다음 흐름으로 처리합니다.

1. 부모 그래프가 **LLM 구조화 출력**으로 문의를 읽고 `billing`, `delivery`, `account`, `general` 중 하나로 분류합니다.
2. 결제/배송/계정 문의는 각각의 **전문가 서브그래프**로 보냅니다.
3. 전문가 서브그래프는 내부 전용 `policy` 키를 사용해 답변을 만들지만, 부모 그래프에는 공유 키인 `answer`만 돌려줍니다.
4. 마지막에는 독립 스키마를 가진 **감사(audit) 서브그래프**를 래퍼 함수로 호출해서 답변 품질을 점검합니다.

```mermaid
flowchart TD
    A["사용자 문의"] --> B["부모: LLM classify_request"]
    B --> C{"category"}
    C -->|billing| D["결제 전문가 서브그래프"]
    C -->|delivery| E["배송 전문가 서브그래프"]
    C -->|account| F["계정 전문가 서브그래프"]
    C -->|general| G["부모: general_answer"]
    D --> H["부모: quality_check wrapper"]
    E --> H
    F --> H
    G --> H
    H --> I["감사 서브그래프"]
    I --> J["최종 답변"]
```


### 실행 방법

아래 셀들을 순서대로 실행하세요.

실행 전 `.env`에 `OPENAI_API_KEY`가 준비되어 있어야 합니다. 이 예제는 라우팅 판단 자체를 LLM에게 맡기므로 샘플 문의 실행마다 모델 호출이 발생합니다.

1. 부모 그래프와 전문가 서브그래프가 공유할 State를 정의합니다.
2. 결제/배송/계정 전문가 서브그래프를 팩토리 함수로 만듭니다.
3. 독립 스키마를 가진 감사 서브그래프를 만듭니다.
4. 부모 그래프에서 LLM 라우팅 결과를 조건부 엣지로 연결해 전문가 서브그래프를 선택합니다.
5. 샘플 질문을 실행한 뒤, 마지막 셀에서 `my_question` 값을 바꿔 직접 테스트해 봅니다.

예시 입력:

```text
배송이 너무 늦어요.
결제했는데 영수증을 못 받았어요.
로그인이 안 되고 비밀번호 재설정 메일도 안 와요.
LangGraph 수업 자료는 어디서 볼 수 있나요?
```


In [ ]:
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import START, END, StateGraph

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 1단계: import + 부모/서브그래프 State 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 2단계 — 전문가 서브그래프 만들기

세 전문가 그래프는 구조가 같습니다.

- `load_policy`: 팀별 처리 기준을 내부 전용 키 `policy`에 저장합니다.
- `compose_answer`: `policy`를 참고해 고객 답변을 작성합니다.

부모 그래프는 이 내부 `policy` 키를 몰라도 됩니다. 부모와 서브그래프가 공유하는 키(`user_text`, `category`, `answer`, `trace`)만 통로처럼 오갑니다.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 2단계: 결제/배송/계정 전문가 서브그래프 팩토리
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3단계 — 독립 스키마 감사 서브그래프 만들기

이번에는 부모 그래프와 키 이름이 다른 감사 서브그래프를 만듭니다.

- 부모 키: `answer`, `audit_status`, `trace`
- 감사 서브그래프 키: `answer_text`, `status`, `memo`

키가 다르기 때문에 이 그래프는 부모에 직접 추가하지 않고, 나중에 `quality_check` 래퍼 함수 안에서 `audit_graph.invoke(...)`로 호출합니다.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3단계: 독립 스키마를 가진 감사(audit) 서브그래프
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4단계 — 부모 그래프 조립하기

부모 그래프의 역할은 오케스트레이션입니다.

1. `classify_request`가 **LLM 구조화 출력**으로 문의의 담당 팀, 근거, 신뢰도를 반환합니다.
2. `route_to_team` 조건부 엣지가 LLM이 고른 카테고리에 맞는 전문가 서브그래프로 보냅니다.
3. 전문가 서브그래프 또는 일반 답변 노드가 답변을 작성합니다.
4. `quality_check` 래퍼 함수가 독립 스키마 감사 서브그래프를 호출합니다.


In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

support_router_model = init_chat_model("openai:gpt-4o-mini", temperature=0)

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 4단계: 부모 그래프의 LLM 라우터 노드와 라우팅 함수 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 5단계: 부모 그래프 조립 + 컴파일
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5단계 — 그래프 구조 확인하기

`xray=True`로 보면 부모 그래프 안에 들어간 전문가 서브그래프의 내부 노드까지 펼쳐서 볼 수 있습니다.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: LLM classify_request → 전문가 서브그래프/일반 답변 → quality_check → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 6단계 — 샘플 문의로 실행 흐름 추적하기

`subgraphs=True`를 켜고 스트리밍하면 부모 그래프 이벤트와 서브그래프 내부 이벤트를 함께 볼 수 있습니다. 네임스페이스가 비어 있으면 부모 그래프, 값이 있으면 특정 서브그래프 내부 이벤트입니다.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 6단계: 실행 도우미 함수 + 샘플 질문 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 7단계 — 직접 문의를 바꿔 실행하기

마지막으로 `my_question` 문자열만 바꿔서 라우팅 결과가 어떻게 달라지는지 확인해 보세요. 이 셀은 별도의 옵션 분기 없이, 입력 문장 하나가 바로 LLM 라우터와 서브그래프 파이프라인을 통과하도록 구성했습니다.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 7단계: 직접 문의를 바꿔 실행하기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **공유 키 패턴 (Case 1)**: 부모와 서브그래프가 동일한 키를 공유하면 컴파일된 서브그래프를 `add_node`에 직접 전달할 수 있어요. LangGraph가 공유 키를 통해 상태를 자동으로 주고받아요.
- **독립 스키마 패턴 (Case 2)**: 스키마가 완전히 다른 경우, 래퍼(wrapper) 함수 안에서 `subgraph.invoke()`를 호출하고 상태를 수동으로 변환해요.
- **3계층 중첩 구조**: grandchild → child → parent 구조도 동일한 래퍼 패턴을 반복 적용해요. 각 계층은 바로 아래 계층만 알면 되고, 더 깊은 계층은 몰라도 돼요.
- **프라이빗 상태(Private State)**: 서브그래프 전용 키는 부모 그래프로 전파되지 않아요. 내부 처리용 임시 키를 안전하게 격리할 수 있어요.
- **subgraphs=True 스트리밍**: 중첩된 서브그래프의 모든 내부 단계를 추적할 수 있어요. 네임스페이스 튜플로 어느 계층에서 발생한 이벤트인지 구분해요.
- **실무 라우터 예제**: LLM 구조화 출력으로 여러 전문가 서브그래프를 선택하고, 독립 스키마 감사 서브그래프를 래퍼 함수로 호출해 실제 업무 흐름에 가까운 패턴을 구현했어요.


## 다음 노트북 예고

다음 `03-Branching-Parallel.ipynb`에서는 **병렬 분기와 Fan-out/Fan-in 패턴**을 배워요. `Annotated[list, operator.add]` reducer로 여러 노드의 결과를 안전하게 합치는 방법, 조건부 분기로 상황별 다른 경로를 선택하는 방법을 다뤄요. 이번에 만든 서브그래프를 병렬로 여러 개 실행하고 결과를 모으는 고급 패턴까지 이어져요.
